Lab 1. Fake news detection (binary classification)

In [146]:
import pandas as pd

df = pd.read_csv('/Users/e.baronov/Programming/Materials/laboratory works/lab_1/fake_or_real_news.csv')
df.head(15)

,Unnamed: 0,title,text,label
0,8476,You Can Smell Hillary’s Fear,"Daniel Greenfield, a Shillman Journalism Fello...",FAKE
1,10294,Watch The Exact Moment Paul Ryan Committed Pol...,Google Pinterest Digg Linkedin Reddit Stumbleu...,FAKE
2,3608,Kerry to go to Paris in gesture of sympathy,U.S. Secretary of State John F. Kerry said Mon...,REAL
3,10142,Bernie supporters on Twitter erupt in anger ag...,"— Kaydee King (@KaydeeKing) November 9, 2016 T...",FAKE
4,875,The Battle of New York: Why This Primary Matters,It's primary day in New York and front-runners...,REAL
5,6903,"Tehran, USA","\nI’m not an immigrant, but my grandparents ...",FAKE
6,7341,Girl Horrified At What She Watches Boyfriend D...,"Share This Baylee Luciani (left), Screenshot o...",FAKE
7,95,‘Britain’s Schindler’ Dies at 106,A Czech stockbroker who saved more than 650 Je...,REAL
8,4869,Fact check: Trump and Clinton at the 'commande...,Hillary Clinton and Donald Trump made some ina...,REAL
9,2909,Iran reportedly makes new push for uranium con...,Iranian negotiators reportedly have made a las...,REAL


# EDA

## Dataset overview: check size, number of classes, balance, average text length.

In [147]:
# size
df.shape

(6335, 4)

In [148]:
# number of classes and balance
df['label'].value_counts()

label
REAL    3171
FAKE    3164
Name: count, dtype: int64

In [164]:
# proportion
df['label'].value_counts(normalize=True)
df['label'].value_counts(normalize=True)

label
REAL    0.500552
FAKE    0.499448
Name: proportion, dtype: float64

У нас действительно два лейбла с практически идеальным распределением.
Теперь посмотрим на дубликаты остальных столбцов

In [191]:
print(df['Unnamed: 0'].duplicated().sum())
print(df['title'].duplicated().sum())
print(df['text'].duplicated().sum())
print(df.duplicated(['title', 'text']).sum())

0
79
275
29


Интересно, одинаковые ли лейблы внутри дубликатов

In [192]:
print(df.duplicated(['title', 'text', 'label']).sum())
print(df.duplicated(['title', 'label']).sum())
print(df.duplicated(['text', 'label']).sum())
# Количество не уменьшилось, поэтому можно считать, что одинаковые

29
79
275


* Проанализировать дубликаты больше (их разное количество - посмотреть на них и тд, те у которых одинаковое одно - как выглядит другое?)

In [182]:
def avg_statistics(series: pd.Series):
    series_name = series.name
    awc = series.str.split().str.len().mean()
    asl = series.str.len().mean()
    awl = (asl - awc + 1) / awc

    return (
        (f'average words count in {series_name}', awc),
        (f'average symbols length in {series_name}', asl),
        (f'average words length in {series_name}', awl)
    )

In [130]:
avg_statistics(df['title'])

(('average words count in title', np.float64(10.496448303078138)),
 ('average symbols length in title', np.float64(65.2776637726914)),
 ('average words length in title', np.float64(5.314294307842695)))

In [131]:
avg_statistics(df['text'])


(('average words count in text', np.float64(776.3007103393844)),
 ('average symbols length in text', np.float64(4707.250355169692)),
 ('average words length in text', np.float64(5.064982670325436)))

Может быть, есть смысл сравнить эти показатели у разных маркировок

In [153]:
fake_df = df[df['label'] == 'FAKE'].drop('label', axis=1)
real_df = df[df['label'] == 'REAL'].drop('label', axis=1)

fake_title_st = avg_statistics(fake_df['title'])
fake_text_st = avg_statistics(fake_df['text'])
real_title_st = avg_statistics(real_df['title'])
real_text_st = avg_statistics(real_df['text'])

In [154]:
import pandas as pd

df_fr = pd.DataFrame(index=[fake_title_st[i][0] for i in range(3)] +
                           [fake_text_st[i][0] for i in range(3)],
                     columns=['fake', 'real'])

for ft, rt in zip(fake_title_st, real_title_st):
    df_fr.loc[ft[0], 'fake'] = round(float(ft[1]), 2)
    df_fr.loc[rt[0], 'real'] = round(float(rt[1]), 2)

for ft, rt in zip(fake_text_st, real_text_st):
    df_fr.loc[ft[0], 'fake'] = round(float(ft[1]), 2)
    df_fr.loc[rt[0], 'real'] = round(float(rt[1]), 2)

print(df_fr)

                                    fake     real
average words count in title       11.13     9.86
average symbols length in title    69.18    61.38
average words length in title        5.3     5.33
average words count in text       679.13   873.26
average symbols length in text   4121.05  5292.16
average words length in text        5.07     5.06


In [162]:
words_title_diff = ((11.13 - 9.86) / 9.86) * 100
symbols_title_diff = ((69.18 - 61.38) / 61.38) * 100
wordlen_title_diff = ((5.3 - 5.33) / 5.33) * 100

words_text_diff = ((679.13 - 873.26) / 873.26) * 100
symbols_text_diff = ((4121.05 - 5292.16) / 5292.16) * 100
wordlen_text_diff = ((5.07 - 5.06) / 5.06) * 100

При практически одинаковой длине слова мы видим разницу в 12 % между количеством слов и символов, где у fake больше, при этом есть разница около 22% у количества слов и символов, где у real больше. (Отсюда можем поставить гипотезу, что фейковые новости в среднем содержат больше информации для привлечения внимания, а также гипотезу, что настоящие новости соддержат больше деталей, фактов и некоторого контекста.)

In [163]:
print(words_title_diff)
print(symbols_title_diff)
print(wordlen_title_diff)
print(words_text_diff)
print(symbols_text_diff)
print(wordlen_text_diff)

12.880324543610563
12.707722385141745
-0.5628517823639821
-22.230492636786295
-22.129149534405606
0.19762845849803706


In [165]:
df['Unnamed: 0'].unique()

array([ 8476, 10294,  3608, ...,  8622,  4021,  4330], shape=(6335,))

In [168]:
fake_df.sample(10)

,Unnamed: 0,title,text
1851,5752,America’s Senator Jeff Sessions Warns of Worse...,
1197,8927,Decorated ‘Hero’ Cop Caught Using His Authorit...,Home / Be The Change / Government Corruption /...
5947,9882,Trump is The Lesser Evil Because Hes Such a N...,Trump is The Lesser Evil Because Hes Such a ...
3645,9668,Anti-Communist Group Makes Their First Ever En...,UNREAL: Calif. Soldiers Billed for Thousands A...
4956,7078,Steven Seagal receives Russian citizenship on ...,Steven Seagal receives Russian citizenship on ...
5543,5403,Russia Is Hoarding Gold at Breakneck Pace — Th...,Citizen journalism with a punch Russia Is Hoar...
5805,10322,Eric Trump: A candidate under investigation is...,The views expressed herein are the views of th...
2901,8352,US General: Warplanes Will Kill Fleeing ISIS F...,Iraqi Govt Warns Civilians Against Fleeing Mos...
1517,7761,FBI debunks Hillary's Conspiracy Theory: Trump...,\nFBI officials say their investigation into l...
208,10555,Clinton Vs. Trump: Latest Electoral Prediction...,(Before It's News)\nIt is fun to look at polls...


In [169]:
real_df.sample(10)

,Unnamed: 0,title,text
4366,537,"In Iowa, potential candidates compete for 2016...","(CNN) Politicians, journalists and conservativ..."
3587,3842,Obama delivers emotional eulogy for Beau Biden,"Wilmington, Delaware (CNN) President Barack Ob..."
3017,4928,Clinton Enters Fall With Key Advantages in Whi...,"Two months from Election Day, Hillary Clinton ..."
4153,3890,Obama to Make Landmark Presidential Trip to Fa...,Barack Obama will make a long-awaited trip to ...
1616,4582,"Protesting Donald Trump’s Election, Not Wars, ...",Protests and vigils have erupted in major citi...
2036,3572,The real reasons Iran is so committed to its n...,As the deadlines near for Iran and world power...
1360,492,Cloudy economy rains on Obama's parade,"According to a transition pool report, the med..."
4225,3488,Congressional Republicans declare Obama’s budg...,Republican members of Congress on Tuesday decl...
1702,3673,Attacks on abortion providers have increased s...,"A white male gunman killed three people, inclu..."
1004,72,Eleven States Sue Obama Admin. over Transgende...,"Texas Attorney General Ken Paxton, along with ..."


Невозможно корректно проанализировать корректность разметки на лейбл, посмотреть просто полные заголовки и тексты
* заглавные буквы, знаки препинания есть (можно посмотреть как ! влияет на ложь/правду)?

дальше идут дубликаты, они выше
* после дубликатов проверить на пустые значения
```
isnull()
```
* визуализация длин текстов - именнно гистограмма?
тогда уж визуализировать все что имеет значение в количественных данных из того что было

* также посмотреть на минимальную максимальные значения перед визуализацией
